# vLLM smoke test — Qwen3-4B (no quantization)

Runs `Qwen/Qwen3-4B` (full-precision bf16) through **vLLM** offline inference on a single prompt. vLLM (0.16.0) lives in the main project env alongside torch 2.9.1, so use the normal `tam` kernel.

**Recommended sampling settings** (from the Qwen3 model card):

| mode | temperature | top_p | top_k | min_p |
|---|---|---|---|---|
| non-thinking | 0.7 | 0.8 | 20 | 0 |
| thinking | 0.6 | 0.95 | 20 | 0 |

Qwen warns: **do not use greedy decoding** in thinking mode (causes repetition). Native context is 32,768 tokens (YaRN needed beyond that).

In [1]:
# Import tamart FIRST so it sets HF_HOME (-> data/hf) before vllm/transformers
# load. This reuses the already-cached Qwen/Qwen3-4B weights.
import tamart  # noqa: F401
import time

import vllm
from vllm import LLM, SamplingParams

print("vllm", vllm.__version__)

vllm 0.16.0


In [2]:
# gpu_memory_utilization is the fraction of TOTAL GPU memory vLLM reserves for
# weights + KV cache. Keep it modest on a SHARED GPU (other processes/kernels
# may hold memory) or vLLM will fail to allocate. Raise toward 0.9 if the GPU
# is yours alone. max_model_len caps the KV-cache reservation.
llm = LLM(
    model="Qwen/Qwen3-4B",
    dtype="bfloat16",
    gpu_memory_utilization=0.5,
    max_model_len=8192,
)

INFO 05-20 13:41:58 [utils.py:223] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B'}
INFO 05-20 13:41:59 [model.py:529] Resolved architecture: Qwen3ForCausalLM
INFO 05-20 13:41:59 [model.py:1549] Using max model len 8192
INFO 05-20 13:41:59 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-20 13:41:59 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:00 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_e

[W520 13:42:03.164842976 socket.cpp:767] [c10d] The client socket cannot be initialized to connect to [vesta-k18-25]:39771 (errno: 97 - Address family not supported by protocol).


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:04 [gpu_model_runner.py:4124] Starting to load model Qwen/Qwen3-4B...
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:05 [cuda.py:367] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:40 [default_loader.py:293] Loading weights took 34.08 seconds
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:40 [gpu_model_runner.py:4221] Model loading took 7.56 GiB memory and 35.424536 seconds
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:48 [backends.py:916] Using cache directory: /home/nifane/.cache/vllm/torch_compile_cache/bd3054cfd4/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:48 [backends.py:976] Dynamo bytecode transform time: 7.13 s
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:54 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.951 s
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:54 [monitor.py:34] torch.compile takes 9.08 s in total
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:55 [gpu_worker.py:373] Available KV cache memory: 30.61 GiB
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:55 [kv_cache_utils.py:1307] GPU KV cache size: 222,

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 24.17it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 22.21it/s]


(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:59 [gpu_model_runner.py:5246] Graph capturing finished in 4 secs, took 0.53 GiB
(EngineCore_DP0 pid=3038799) INFO 05-20 13:42:59 [core.py:278] init engine (profile, create kv cache, warmup model) took 18.80 seconds
INFO 05-20 13:43:01 [llm.py:355] Supported tasks: ['generate']


In [ ]:
# Non-thinking mode + Qwen's recommended sampling for it.
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    max_tokens=512,
)

messages = [
    {"role": "user", "content": "Give me a short introduction to large language models."}
]

t0 = time.time()
outputs = llm.chat(
    messages,
    sampling_params,
    chat_template_kwargs={"enable_thinking": False},  # non-thinking mode
)
dt = time.time() - t0

out = outputs[0].outputs[0]
n_new = len(out.token_ids)
print(f"generated {n_new} tokens in {dt:.1f}s -> {n_new / dt:.1f} tok/s\n")
print("content:", out.text)

INFO 05-20 13:43:02 [hf.py:318] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

generated 137 tokens in 2.7s -> 50.8 tok/s

content: Large language models (LLMs) are advanced artificial intelligence systems designed to understand and generate human-like text. They are trained on vast amounts of text data from the internet, books, articles, and other sources, allowing them to recognize patterns, context, and meaning in language. LLMs can perform a wide range of tasks, such as answering questions, writing stories, coding, translating languages, and even engaging in complex conversations. These models are typically based on deep learning techniques, using neural networks with many layers to process and generate text. They have revolutionized fields like natural language processing, customer service, content creation, and more, offering powerful tools for both individuals and businesses.


ERROR 05-20 13:50:11 [core_client.py:616] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
